In [1]:
import pytest
import ipytest
from phy import *

In [2]:
ipytest.autoconfig()

In [3]:
%%ipytest

@pytest.fixture
def master_design():
    return {
        "environment": {
            "dimensions": [5, 5, 3],
            "wall_resolution": [2, 2],
            "reflectivity": {"floor": 0.2, "ceiling": 0.8, "walls": 0.5}
        },
        "nodes": {
            "sensors": {
                "positions": np.array([[2, 2, 0]]),
                "rx_type": [0],
                "uplink_type": [0],
                "rx_area": 1e-4,
                "tx_power": 0.5,
                "BW": 1e6,
                "VLC_pass_filter": True,
                "m": 1,
                "nT": [[0,0,1]],
                "nR": [[0,0,1]],
                "IR_tx_power": 0.1,
                "RF_tx_power": 0.1
            },
            "masters": {
                "positions": np.array([[2.5, 2.5, 3]]),
                "rx_area": 1e-3,
                "tx_power": 10.0,
                "sensitivity": -90,
                "BW": 1e6,
                "m": 1,
                "nT": [[0,0,-1]],
                "nR": [[0,0,-1]],
                "uplink_type": 0 # Masters usually act as RX for uplink
            },
            "ambient_nodes": {
                "positions": np.array([[0, 0, 3]]),
                "tx_power": 0.0,
                "m": 1,
                "nT": [[0,0,-1]]
            }
        },
        "TIA": {
            "RF": 1e4, "Vn": 1e-9, "In": 1e-15, 
            "fncV": 100, "fncI": 100, "temperature": 298
        }
    }
# --- 1. Initialization and Integration ---

def test_phynet_init_and_noise(master_design):
    """Verify kernel initializes sub-components and computes noise current."""
    net = PhyNet(master_design)
    
    # Check if managers were created
    assert isinstance(net.snm, SNManager)
    assert isinstance(net.mnm, MNManager)
    
    # Check if noise was calculated
    # Since we have a PD receiver, x_d_noise (current variance) should exist
    assert net.x_d_noise is not None
    assert net.x_d_noise[0] > 0

# --- 2. Link Budget & Power Optimization ---

def test_phynet_power_optimization(master_design):
    """Test the 'budget_run' feature that calculates minimum required Tx power."""
    # Run with budget_run=True
    net = PhyNet(master_design, budget_run=True)
    
    # Verify that p_req (Required Tx Power) was calculated
    assert hasattr(net, 'p_req')
    assert net.p_req[0] > 0
    
    # Verify that the sensor Tx power was updated from default (0.5) to optimized
    assert net.snm.OTx_elements.p == net.p_req

# --- 3. Performance Metrics (SNR/BER) ---

def test_phynet_metrics_pd(master_design):
    """Verify SNR and BER calculation for Photodiode nodes."""
    net = PhyNet(master_design)
    
    # SNR should be a positive number
    assert net.snr_d[0] > 0
    # BER should be between 0 and 0.5
    assert 0 <= net.BER_d[0] <= 0.5

def test_phynet_pv_handling(master_design):
    """Force a PV receiver and ensure the PV circuit model runs."""
    # Modify design to use PV
    master_design["nodes"]["sensors"]["rx_type"] = 1 
    
    net = PhyNet(master_design)
    
    # Verify that the PV object (pvx) was instantiated
    assert hasattr(net, 'pvx')
    assert isinstance(net.pvx, PV)
    # Check PV-specific SNR
    assert net.snr_pv > 0

....                                                                                         [100%]
4 passed in 0.09s
